# Notebook 8 / 8 — Experimental: Composing Non-Classical Logics in Language Tasks

> **Series.** This is the eighth and final (experimental) notebook.
> 1. *Basics* — eight foundational logics
> 2. *Advanced* — fourteen rarer logics
> 3. *Synthesis* — cross-logic benchmarks and conclusions
> 4. *Applications* — ten general-purpose domains
> 5. *Language* — non-classical logics in NLP tasks
> 6. *Workflow* — an end-to-end pipeline composing the best logics
> 7. *LangGraph* — the same pipeline as a state machine
> 8. **Experimental composition** *(this notebook)* — probing what happens when two non-classical logics *overlap* on the same linguistic phenomenon

## What makes this one *experimental*

Notebook 5 used each logic on a disjoint language task (fuzzy for vagueness, LP for contradictions, free logic for presuppositions, …). Notebook 6/7 composed them as *pipeline stages*, each owning a different slice of state. That composition was safe precisely because the slices did **not** overlap.

This notebook asks the harder question: **what if two logics want to speak about the same atom at the same time?** Natural language rarely respects the disjoint-slice discipline — a single sentence can simultaneously be *vague* AND *contradicted by another speaker* AND *have a non-denoting subject* AND *carry a default implicature*. The interesting research question is which compositions behave and which ones explode.

We run six **experiments**, each probing a specific overlap:

| # | Overlap being probed | Logics that collide | What we measure |
|---|---|---|---|
| 1 | Vagueness **under** contradiction | Fuzzy × LP | Does a fuzzy degree survive when speakers also contradict each other? |
| 2 | Presupposition failure **inside** a hedged report | Free × Possibilistic | How does a necessity weight propagate through `*` (undefined)? |
| 3 | Defaults that fire on **contradicted** premises | Default × LP | Should a default still trigger if its precondition is in state `{T,F}`? |
| 4 | Tense operators **over** intuitionistic claims | LTL × Intuitionistic | Does `F φ` require a *current* witness or a *future* witness? |
| 5 | Annotator disagreement **on** a vague label | Subjective × Fuzzy | Two sources of gradient — do they compose or fight? |
| 6 | Nested beliefs **about** a word's sense | Epistemic × Dempster–Shafer | Can Alice know that Bob assigns a *different subset* of senses? |

Each experiment is a self-contained cell with: a minimal data model, a *collision hypothesis*, and a small empirical observation. At the end we extract **composition laws** — patterns that work, patterns that don't.

## Conventions for the whole notebook

- Every experiment prints a short *hypothesis* first, then the measurement.
- Data types are reused across experiments where sensible; where a collision requires a new join type, we give it a name (`FuzzyLP`, `FreePoss`, etc.) so it is clear *that a new object exists*.
- We avoid any "obvious" reduction — the point is not to show that logics agree, but to find the seams.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Set, Tuple, Optional, FrozenSet, Callable
from itertools import product
from functools import reduce
import math

def hypothesis(text):
    print("\n" + "─"*64)
    print("HYPOTHESIS:", text)
    print("─"*64)

def observation(text):
    print("OBSERVATION:", text)

## Experiment 1 — Vagueness under contradiction (Fuzzy × LP)

**Linguistic setup.** Two speakers describe the same man:
- Alice: "He is tall."
- Bob: "He is *not* tall."

Each speaker's assertion is itself fuzzy — they are not asserting a sharp boolean, they are asserting a *degree*. The man is 1.80 m, so `tall(1.80) ≈ 0.57`. Alice endorses the positive pole, Bob the negative.

**Collision hypothesis.** A naive composition would say: *Alice and Bob contradict each other, so use LP and return `{T,F}`.* But the fuzzy degree is *already* half-and-half — assigning `{T,F}` erases what fuzzy already told us.

The proposal tested here: lift to a **FuzzyLP** value `(μ_T, μ_F)` where both components live in `[0,1]`. Alice pushes `μ_T` up, Bob pushes `μ_F` up, and *designation* ("should I act on this?") requires `μ_T > μ_F + ε`.

In [ ]:
@dataclass
class FuzzyLP:
    mu_T: float   # degree of positive support
    mu_F: float   # degree of negative support

    def status(self, eps=0.15):
        gap = self.mu_T - self.mu_F
        if  gap >  eps: return "designated (lean positive)"
        if -gap >  eps: return "designated (lean negative)"
        return "contested — neither pole dominates"

def fuzzy_tall(height_cm):
    if height_cm <= 160: return 0.0
    if height_cm >= 195: return 1.0
    return (height_cm - 160) / 35

hypothesis("A contradicted vague predicate should yield a *gap*, not a single value.")

base = fuzzy_tall(180)
# Alice asserts tall at full confidence; Bob asserts not-tall at full confidence.
# Each endorses their pole *up to the underlying fuzzy degree*.
v = FuzzyLP(mu_T=base, mu_F=1.0 - base)
print(f"  man is 180 cm → fuzzy tall = {base:.2f}")
print(f"  combined FuzzyLP = (mu_T={v.mu_T:.2f}, mu_F={v.mu_F:.2f})")
print(f"  status: {v.status()}")

# What happens at 192 cm, where fuzzy tall = 0.91?
base2 = fuzzy_tall(192)
v2 = FuzzyLP(mu_T=base2, mu_F=1 - base2)
print(f"\n  man is 192 cm → fuzzy tall = {base2:.2f}")
print(f"  combined FuzzyLP = (mu_T={v2.mu_T:.2f}, mu_F={v2.mu_F:.2f})")
print(f"  status: {v2.status()}")

observation("At 180 cm, Alice and Bob are both *partially right* — gap ≈ 0.14 → contested.")
observation("At 192 cm, Alice's assertion is mostly witnessed by the fuzzy degree — Bob is simply wrong at that height.")
observation("Flattening either dimension loses information that speakers track.")

## Experiment 2 — Presupposition failure inside a hedged report (Free × Possibilistic)

**Linguistic setup.** A news wire reports: *"The prime minister's spouse allegedly denied the allegations."* The sentence carries a weight (the hedge *allegedly*, worth ≈ 0.7 in a possibilistic KB) **and** a presupposition (there is such a spouse). Suppose the prime minister is single.

**Collision hypothesis.** Possibilistic logic wants to answer `necessity = 0.7`. Free logic wants to answer `*` (presupposition failure). If the composition forces a single number, the hedge dominates and the presupposition failure disappears.

The proposal: a *lexicographic* ordering — free-logic's `*` shortcircuits any weight, but if the presupposition holds, the possibilistic weight propagates normally. The composite type is a **Maybe[float]** with three states: `(defined, w)`, `(undef, ·)`, and an explicit `*` tag.

In [ ]:
@dataclass
class FreePoss:
    weight: Optional[float]     # None = presupposition failure
    reason: str                  # human-readable tag

    def __str__(self):
        if self.weight is None: return f"* ({self.reason})"
        return f"necessity {self.weight:.2f} ({self.reason})"

REFERENCES = {
    "the_pm_spouse":      None,              # PM is single
    "the_minister_aide":  "aide_smith",
}
HEDGE = {"allegedly": 0.7, "reportedly": 0.85, None: 1.0}

def interpret(subject_desc: str, hedge: Optional[str]) -> FreePoss:
    referent = REFERENCES.get(subject_desc)
    if referent is None:
        return FreePoss(weight=None, reason=f"no referent for '{subject_desc}'")
    return FreePoss(weight=HEDGE[hedge], reason=f"refers to {referent}")

hypothesis("Presupposition failure should shortcircuit hedge weight, not be averaged into it.")

print("  'The PM's spouse allegedly denied' →", interpret("the_pm_spouse", "allegedly"))
print("  'The minister's aide allegedly denied' →", interpret("the_minister_aide", "allegedly"))
print("  'The minister's aide denied'            →", interpret("the_minister_aide", None))

observation("A composite type is *required*. You cannot represent 'there is no such person, but if there were we would trust the report at 0.7' in a single float.")
observation("The pipeline must also branch on `*` explicitly — downstream code that just tests `weight > 0.5` would quietly accept the nonsense.")

## Experiment 3 — Defaults firing on contradicted premises (Default × LP)

**Linguistic setup.** A dialogue system maintains a discourse representation in LP (so contradictions stay contained). One default rule says *"birds typically fly"*. A speaker asserts `bird(tweety)`; another speaker asserts `¬bird(tweety)` (maybe the user mis-identifies an animal). The predicate `bird(tweety)` is now LP-valued `{T,F}`.

**Collision hypothesis.** Classical default logic would fire on `bird(tweety) = T` and conclude `flies(tweety)`. But here `bird(tweety)` is *contested*. Should the default still fire?

We compare three composition policies:
- **Strict.** Default fires only if the precondition is `{T}` (fully positive, uncontested).
- **Credulous.** Default fires if `T` is in the precondition's value — i.e. `{T}` OR `{T,F}`.
- **Skeptical-LP.** Default fires, but the conclusion inherits the LP status: if the premise is `{T,F}`, the conclusion is `{T,F}` too — it gets to propagate the contest.

In [ ]:
Tv = frozenset({"T"}); Fv = frozenset({"F"}); Bv = frozenset({"T","F"}); Nv = frozenset()

@dataclass
class LPDefault:
    pre: str
    justification: str
    conclusion: str

def apply_default(kb: Dict[str, FrozenSet[str]], d: LPDefault, policy: str):
    pre_val = kb.get(d.pre, Nv)
    just_blocked = ("not_"+d.justification) in kb and "T" in kb["not_"+d.justification]

    if policy == "strict":
        if pre_val == Tv and not just_blocked:
            kb[d.conclusion] = Tv
    elif policy == "credulous":
        if "T" in pre_val and not just_blocked:
            kb[d.conclusion] = Tv
    elif policy == "skeptical_lp":
        if "T" in pre_val and not just_blocked:
            kb[d.conclusion] = pre_val       # inherit LP status
    return kb

default = LPDefault("bird", "flies", "flies")

hypothesis("A default on a contested premise should propagate the contest, not pretend the premise is settled.")

for policy in ("strict", "credulous", "skeptical_lp"):
    kb = {"bird": Bv}   # speakers disagreed whether Tweety is a bird
    apply_default(kb, default, policy)
    print(f"  policy={policy:14s}  flies = {set(kb.get('flies', Nv)) or '— (default did not fire)'}")

print()
for policy in ("strict", "credulous", "skeptical_lp"):
    kb = {"bird": Tv}   # uncontested
    apply_default(kb, default, policy)
    print(f"  uncontested — policy={policy:14s}  flies = {set(kb.get('flies', Nv)) or '—'}")

observation("`strict` silently swallows the contested case; downstream cannot tell whether the absence of `flies` is a rebuttal or a non-observation.")
observation("`credulous` re-asserts `flies = T` — dangerous, because downstream loses the fact that the premise is still disputed.")
observation("`skeptical_lp` propagates `flies = {T,F}`, preserving the audit trail. This is the only policy that survives chaining with further defaults.")

## Experiment 4 — Tense over intuitionistic claims (LTL × Intuitionistic)

**Linguistic setup.** A news feed arrives one document at a time (like a retrieval stream). At each step, more atoms become *witnessed*. Consider the sentence *"Eventually, a court will rule on the case."* — an LTL `F` claim.

**Collision hypothesis.** LTL's `F φ` normally means "at some point in the trace, `φ` holds". Intuitionistic logic insists `φ` only holds at a state where a *witness exists*. Compose them: `F φ` is satisfied only if the trace contains a state whose witness set forces `φ`. Crucially, at the *current* state this is often neither forced nor refuted — it is *not yet decided*, which is exactly the intuitionistic answer we want.

We define **IntLTL**: a four-step trace where each step is an intuitionistic state. `F φ` becomes "there exists a state `s_i` whose valuation already includes a witness for `φ`"; `G φ` becomes "every future state has a witness".

In [ ]:
# trace[i] = set of atoms that have a constructive witness at step i
trace = [
    set(),                              # t0: nothing known
    {"case_filed"},                     # t1: filing witnessed
    {"case_filed", "hearing_held"},     # t2: hearing witnessed
    {"case_filed", "hearing_held", "court_ruled"},  # t3: ruling witnessed
]

def intltl(f, trace, i=0):
    op = f[0]
    n = len(trace)
    if i >= n: return False
    if op == "atom":  return f[1] in trace[i]          # witnessed at i?
    if op == "F":     return any(intltl(f[1], trace, j) for j in range(i, n))
    if op == "G":     return all(intltl(f[1], trace, j) for j in range(i, n))
    if op == "not":
        # intuitionistic: ¬φ at i iff no future state witnesses φ
        return all(not intltl(f[1], trace, j) for j in range(i, n))
    if op == "and":   return intltl(f[1], trace, i) and intltl(f[2], trace, i)
    if op == "or":    return intltl(f[1], trace, i) or  intltl(f[2], trace, i)
    raise ValueError(op)

hypothesis("'Eventually φ' should be *assertible* at t0 only if the trace already contains a witnessing state — otherwise it is 'not yet decided'.")

for i in range(4):
    print(f"\n  At t{i} (witness set = {trace[i] or '∅'}):")
    print(f"    F court_ruled   assertible? {intltl(('F',('atom','court_ruled')),   trace, i)}")
    print(f"    F hearing_held  assertible? {intltl(('F',('atom','hearing_held')),  trace, i)}")
    print(f"    ¬ F court_ruled assertible? {intltl(('not',('F',('atom','court_ruled'))), trace, i)}")

observation("At t0, `F court_ruled` is already assertible BECAUSE the trace contains t3 — even though at t0 no witness exists locally.")
observation("`¬ F court_ruled` is NOT assertible at t0 — intuitionistic negation checks that no future state witnesses it, and t3 does.")
observation("This is the 'eternal' reading of `F` working together with the 'witness-first' reading of atoms. It is exactly how a journalist reading a closed archive interprets 'will eventually' in past tense.")

## Experiment 5 — Annotator disagreement on a vague label (Subjective × Fuzzy)

**Linguistic setup.** Five human raters score a restaurant review for *positivity* on a 5-point scale. The label *positive* is already fuzzy (graded), and the raters disagree.

**Collision hypothesis.** Two natural aggregates exist:
- Convert each rating to a fuzzy `[0,1]` degree and take the *mean*.
- Convert each rating to a subjective-logic opinion and fuse cumulatively.

These are *not* the same operation. Fuzzy mean erases dispersion; cumulative fusion preserves it as residual uncertainty. On the same input they give different downstream behaviour.

In [ ]:
@dataclass
class Op:
    b: float = 0.0; d: float = 0.0; u: float = 1.0; a: float = 0.5
    def proj(self): return self.b + self.a*self.u

def fuse(o1: Op, o2: Op) -> Op:
    if o1.u == 0 and o2.u == 0:
        return Op((o1.b+o2.b)/2, (o1.d+o2.d)/2, 0.0, o1.a)
    den = o1.u + o2.u - o1.u*o2.u
    return Op(
        (o1.b*o2.u + o2.b*o1.u)/den,
        (o1.d*o2.u + o2.d*o1.u)/den,
        (o1.u*o2.u)/den, o1.a,
    )

def rating_to_fuzzy(r): return (r - 1) / 4      # 1..5 → 0..1
def rating_to_opinion(r, confidence=0.8):
    mu = rating_to_fuzzy(r)
    return Op(b=mu*confidence, d=(1-mu)*confidence, u=1-confidence)

cases = [
    ("clear consensus",       [4, 4, 5, 4, 5]),
    ("polarised (bimodal)",   [5, 5, 5, 1, 1]),
    ("soft middle",           [3, 3, 3, 3, 3]),
]

hypothesis("Fuzzy averaging and subjective fusion agree on unimodal cases and disagree on bimodal ones — and only subjective logic tells you *which* case you are in.")

for label, ratings in cases:
    fuzzy_mean = sum(rating_to_fuzzy(r) for r in ratings) / len(ratings)
    fused = reduce(fuse, (rating_to_opinion(r) for r in ratings))
    print(f"  {label:20s}  fuzzy_mean = {fuzzy_mean:.2f}")
    print(f"  {'':20s}  subjective = b={fused.b:.2f} d={fused.d:.2f} u={fused.u:.2f}  proj={fused.proj():.2f}")

observation("Fuzzy mean reports 0.50 for BOTH 'polarised' and 'soft middle' — indistinguishable.")
observation("Subjective fusion shows polarised = low residual `u` with b≈d (i.e., strong disagreement), vs soft middle = same `u` level but no opposing mass.")
observation("Composition rule: fuzzy encodes the *label's* vagueness; subjective encodes the *raters'* disagreement. They should live in different fields of the same record, not be collapsed together.")

## Experiment 6 — Nested beliefs about a word's sense (Epistemic × Dempster–Shafer)

**Linguistic setup.** *Alice and Bob both see the word* **bank** *in a sentence. Alice thinks Bob assigns it to the financial sense; Bob actually assigns it to the river-bank sense. The agent coordinating them needs to model not just each speaker's sense assignment, but also each speaker's model of the other's assignment.*

**Collision hypothesis.** Epistemic logic operates on propositional atoms. Dempster–Shafer operates on mass functions over subsets of a frame. Composing them means *each epistemic world carries its own mass function*, and `K_alice` quantifies over the worlds Alice considers possible — each of which may have different masses.

We build a miniature *epistemic Dempster–Shafer model*: worlds are tagged with a mass function; `K_i φ` for a proposition `φ` over senses becomes "in every world Alice considers possible, the mass function assigns φ ≥ threshold".

In [ ]:
fs = lambda *xs: frozenset(xs)
Sense = fs("financial", "river")   # simplified frame

# Each world is labelled by Bob's *actual* sense assignment.
# Alice's accessibility relation encodes what Alice thinks is possible.
worlds = {
    "w_fin":   {fs("financial"): 1.0},        # Bob really means financial
    "w_riv":   {fs("river"):     1.0},        # Bob really means river
    "w_mixed": {fs("financial","river"): 1.0} # Bob is himself ambiguous
}

# Alice's model: she is confident it's financial, considers mixed as a fallback.
R_alice = {"actual": ["w_fin", "w_mixed"]}
# Bob's model of himself: he knows he means river.
R_bob   = {"actual": ["w_riv"]}
# The *actual* world of the conversation:
actual = "w_riv"

def belief_in_sense(world: str, sense: str) -> float:
    mass = worlds[world]
    return sum(m for s,m in mass.items() if sense in s)

def K(agent: str, sense: str, threshold: float) -> bool:
    """Agent knows that mass(sense) >= threshold, in every world they consider possible."""
    R = R_alice if agent == "alice" else R_bob
    return all(belief_in_sense(w, sense) >= threshold for w in R["actual"])

hypothesis("Nested belief over sense masses exposes the gap between 'Alice knows Bob's mass' and 'Alice's mass agrees with Bob's mass'.")

print(f"  actual world: {actual} (Bob really means 'river')")
print()
print("  Does Alice know bank = financial (threshold 0.8)?   ", K("alice", "financial", 0.8))
print("  Does Alice know bank = river     (threshold 0.8)?   ", K("alice", "river",     0.8))
print("  Does Bob   know bank = river     (threshold 0.8)?   ", K("bob",   "river",     0.8))

# The critical second-order query: does Alice know that Bob knows?
# It requires: in every world Alice considers possible, Bob's R-set gives him a singular sense assignment.
def K_alice_of_K_bob(sense: str, threshold: float) -> bool:
    return all(
        all(belief_in_sense(bw, sense) >= threshold for bw in R_bob["actual"])
        for aw in R_alice["actual"]
    )

print()
print("  Does Alice know that Bob knows bank = financial? ", K_alice_of_K_bob("financial", 0.8))
print("  Does Alice know that Bob knows bank = river?     ", K_alice_of_K_bob("river",     0.8))

observation("Alice is wrong: she does not know `river`, and she *does* believe Bob means `river` via her fallback world — but only the fallback. Her primary hypothesis is still `financial`.")
observation("The composite type is `World → MassFunction`. You cannot shortcut this with a single mass per agent; the nesting is what makes the linguistic coordination problem visible.")

## Composition laws (derived from the six experiments)

Pulling the observations together:

### Law 1. **No silent collapse.**
When two logics overlap on the same atom, the result almost never fits in a single scalar. Experiments 1, 2, 4, 5 all required a *composite type* (`FuzzyLP`, `FreePoss`, per-step witness set, split fuzzy+opinion). Pipelines that represent evidence as a single float will lose one of the two signals.

### Law 2. **Undefined shortcircuits weight.**
Free logic's `*` must be a first-class case in any downstream numeric operation. Possibilistic weights, subjective projections, and LTL operators all need explicit branches for "undefined" — otherwise the hedge (Experiment 2) looks confident when the referent does not exist.

### Law 3. **Contested premises propagate.**
Defaults (and in general any non-monotonic rule) must inherit the LP status of their premises. The `skeptical_lp` policy in Experiment 3 is the only one that survives being chained — otherwise contest information is silently dropped at each rule application.

### Law 4. **Truth-of-atoms and truth-of-connectives are different axes.**
Experiments 1 and 5 show that vagueness (fuzzy) lives *in the atom* while disagreement (subjective) lives *in the agents* and contradiction (LP) lives *in the discourse*. These are three different axes and the composite record should have three separate fields, not one aggregate score.

### Law 5. **Temporal operators see witness sets, not truth values.**
Experiment 4's `IntLTL` works because `F φ` asks *whether a witness exists somewhere in the trace*, not *whether a classical value is true*. This composition is unusually clean precisely because LTL *only* needs a boolean-shaped answer per step — the intuitionistic layer provides exactly that, without needing to reinterpret the temporal operators.

### Law 6. **Epistemic operators lift whatever lives at worlds.**
Experiment 6 shows that *anything* interpretable at a world can be lifted under `K_i`. In particular, Dempster–Shafer mass functions at worlds give you "agent-specific belief over belief". This is the richest composition in the notebook and also the most expensive to check.

### Law 7. **Composition is easy when the logics own different axes; hard when they fight for the same one.**
A concise restatement of the pattern in Experiments 5 and 6: when each logic owns a distinct dimension (vagueness vs disagreement, worlds vs mass), composition is trivial. When they both want to say something about the same dimension (Experiment 1: both fuzzy and LP want to modulate the *truth* axis of `tall`), you must pick a *product* type, not a quotient.

## Open experimental directions

1. **Fuzzy + LP + possibilistic three-way.** Can a single record carry `(mu_T, mu_F, necessity)` and still support a sound consequence relation?
2. **Intuitionistic negation inside a default justification.** Does the `skeptical_lp` policy generalise if the *justification* `β` is itself intuitionistic?
3. **DEL action models with LP premises.** What happens when the *precondition* of an action model event is contested?
4. **Subjective-logic discounting as a hedge multiplier.** Bridges Experiments 2 and 5: treat the hedge as a *trust* value that discounts each source's opinion.
5. **Empirical calibration.** Collect natural-language examples (news snippets, Reddit threads, medical notes) and annotate them for which axes collide. The composition laws above are conjectures until they are checked against real text.

## Closing thought

Across seven earlier notebooks the lesson was: *pick the smallest fragment that names the detail*. This notebook sharpens it:

> **When two non-classical logics speak about the same phenomenon, the right composite is almost always a *product* — a richer type with one field per axis — and almost never a *quotient* that averages the axes into a single score.**

Most of the visible failures of NLP pipelines under contradiction, hedging, disagreement, or presupposition can be traced to a premature quotient. The remedy is a larger state object and a slightly more expensive check, not a cleverer logic.